# Solar filament segmentation: preprocessing, training (optional), inference, post-processing, evaluation

Solar Filament Segmentation Challenge 2026. This notebook runs the submitted pipeline end to end and rebuilds the
final submission from the weights in `weights/` and the 180 official test H-alpha images. It only calls modules;
the implementation is in `src/solar/`:

| Section | Module | What it does |
| --- | --- | --- |
| 1. Preprocessing | `solar.final_pipeline` | solar-disk geometry and limb-darkening flattening of every image |
| 2. Training (optional) | `solar.final_training` | the recipe of every file in `weights/`; `TRAIN = True` retrains instead of loading |
| 3. Inference | `solar.final_pipeline` | U-Net foreground logits and YOLO11m-seg detections (with test-time flips) |
| 4. Post-processing | `solar.final_pipeline` | fusion, rescorer, instance refiner, submission CSV |
| 5. Evaluation | `solar.final_evaluation` | PQ, IoU and Dice distributions, one-to-many / many-to-one counts on held-out images |

**One pipeline, one switch.** The switch is the detector set; everything else is shared.

| Setting | detectors | rescorer | Kaggle CSV (sha256 prefix) |
| --- | --- | --- | --- |
| `final1_A6` (submitted, default) | cp_s43, m1280, m1024, det4, d1024, d1536 | `weights/rescorer_A6.json` | `2ba22b86` |
| `final2_r0` | cp_s43, m1280, m1024 | `weights/rescorer_r43.json` | `fafef71c` |

Stages and their fixed settings:

1. preprocess: disk geometry + flattening -> `flat/*.png`, `disk.json`
2. U-Net (EfficientNet-B4, 512 px tiles, overlap 128, dihedral-8 TTA) -> float16 logits per image
3. YOLO11m-seg detectors, four flipped copies each (conf .10, iou .60, agnostic NMS, retina masks)
4. fusion (U-Net prob .5, close 3, grow 4 px, min_area 200, candidate floor .15) + frozen logistic rescorer (keep iff p x E[IoU] > PQ_train / 2)
5. instance refiner (3-channel EfficientNet-B4 U-Net, 512 px tiles, dihedral-8, prob .5, 200 px fallback)
6. CSV (COCO RLE, non-overlapping masks) + comparison with the uploaded file when it is available

**Inputs**: the competition data under `data/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/` (`test/test_images/*.jpeg`, 180 files;
the training images and annotations for sections 2 and 5) and `weights/` (verified by SHA-256 first).
**Outputs**: `outputs/final_inference/<setting>/submission.csv` and `outputs/final_inference/report.json` (timings, environment, checks, evaluation).

**Hardware**: one GPU is recommended (the weights were produced on an NVIDIA RTX A5000 24 GB and a GTX 1080 Ti); CPU works but is slow.
The last cell prints the time of every stage. Environment: `requirements-model.txt` (torch 2.6.0+cu124, ultralytics 8.3.221,
segmentation-models-pytorch 0.5.0, ...).

Run headless from the repository root:
```
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 notebooks/final_inference.ipynb --output executed.ipynb
```
Reproducibility note: GPU convolutions (cuDNN autotuning, TF32 on Ampere) are not bit-exact across runs or cards, so the
rebuilt CSV may differ from the uploaded one in a few instances; the comparison reports exact-RLE matches and the share of
instances matched at IoU > .5. Everything after the networks (fusion, rescorer, refiner rules, CSV) is deterministic.

In [1]:
# Parameters (the only cell to edit)
RUN_FINALS = ['final1_A6']   # the submitted setting; add 'final2_r0' to also rebuild the 3-detector setting
LIMIT = None            # smoke test: e.g. 2 -> first two test images only (no byte comparison)
DEVICE = None           # None -> 'cuda:0' when available, else 'cpu'
OUT_DIR = None          # None -> outputs/final_inference
PREPROCESS_WORKERS = 4  # CPU processes for disk detection + flattening
REFINER_BATCH = 16      # tiles per forward pass in the refiner (lower on small GPUs)
COMPARE = True          # compare with the uploaded final CSVs when they exist under outputs/
TRAIN = False           # False: load weights/ (default). True: retrain TRAIN_STEPS first (section 2; many GPU-hours)
TRAIN_STEPS = ['inputs', 'unet', 'cp_s43', 'm1280', 'm1024', 'det4', 'd1024', 'd1536', 'rescorer', 'refiner']
EVALUATE_RERUN = True

In [2]:
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'weights/manifest.json').exists() else Path.cwd().parent
assert (ROOT / 'weights/manifest.json').exists(), ROOT
os.chdir(ROOT)   # the modules use repository-relative data paths
sys.path.insert(0, str(ROOT / 'src'))
try:
    import pandas  # noqa: F401  (Windows: import pandas before cv2 + segmentation_models_pytorch)
except ImportError:
    pass
import torch
import solar.final_pipeline as fp
import solar.final_training as ft
import solar.final_evaluation as fe

DEVICE = DEVICE or ('cuda:0' if torch.cuda.is_available() and torch.cuda.device_count() else 'cpu')
OUT = Path(OUT_DIR) if OUT_DIR else ROOT / 'outputs/final_inference'
OUT.mkdir(parents=True, exist_ok=True)
report = {'started': time.strftime('%Y-%m-%d %H:%M:%S'), 'environment': fp.environment(DEVICE), 'timings': {}, 'stages': {}, 'finals': {}}
report['weights'] = fp.check_weights()            # SHA-256 of every weight == manifest == pinned list
stems = fp.test_stems()                            # 180 official test images (SHA-audited when the manifest exists)
if LIMIT:
    stems = stems[:LIMIT]
IMAGE_DIR = fp.DEFAULT_DATA / 'test/test_images'
CACHE, LOGITS, VIEWS = OUT / 'cache', OUT / 'logits_unet', OUT / 'views'
members = tuple(dict.fromkeys(m for f in RUN_FINALS for m in fp.POOLS[fp.FINALS[f]]))
print(json.dumps(report['environment'], indent=1))
print(f'{len(stems)} test images; finals {RUN_FINALS}; detectors {members}; device {DEVICE}')

/home/<user>/solar_filament_2026/final_check_20261112/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
 "python": "3.11.18",
 "platform": "Linux-5.15.0-139-generic-x86_64-with-glibc2.35",
 "torch": "2.6.0+cu124",
 "cuda": "12.4",
 "cudnn": 90100,
 "device": "cuda:0",
 "cpu_count": 64,
 "tf32_matmul": false,
 "tf32_cudnn": true,
 "gpu": "NVIDIA RTX A5000",
 "gpu_memory_gb": 23.7,
 "numpy": "2.0.0",
 "cv2": "4.11.0",
 "pycocotools": "present",
 "segmentation_models_pytorch": "0.5.0",
 "timm": "1.0.29",
 "ultralytics": "8.3.221",
 "albumentations": "2.0.8",
 "sklearn": "1.7.2"
}
180 test images; finals ['final1_A6']; detectors ('cp_s43', 'm1280', 'm1024', 'det4', 'd1024', 'd1536'); device cuda:0


## 1. Preprocessing

Solar-disk geometry and limb-darkening flattening of the test images (CPU). The U-Net, three of the
detectors and the refiner read the flattened images; the other three detectors read the raw JPEG.

In [3]:
# 1. preprocess (CPU): disk geometry + limb-darkening flattening
t = time.perf_counter()
report['stages']['preprocess'] = fp.preprocess(stems, IMAGE_DIR, CACHE, workers=PREPROCESS_WORKERS)
report['timings']['preprocess_s'] = time.perf_counter() - t
print(report['stages']['preprocess'], f"{report['timings']['preprocess_s']:.0f}s")

{'images': 180, 'computed': 180, 'cache_dir': '/home/<user>/solar_filament_2026/final_check_20261112/repo/outputs/final_inference/cache'} 29s


## 2. Training (optional)

By default nothing is trained: the stages below load `weights/`. The cell prints how each file was produced.
With `TRAIN = True` it calls `solar.final_training.retrain`, which runs the chosen `TRAIN_STEPS` in order
(inputs -> U-Net -> six detectors -> rescorer -> refiner), writes a complete set of weights to
`outputs/retrain/weights/` and points the rest of the notebook at it. Steps that are left out keep the shipped file.

- Training uses only the competition's annotations. Starting points are public ImageNet (EfficientNet-B4) and COCO
  (yolo11m-seg) weights, pinned by SHA-256 and downloaded by the `inputs` step.
- Split: observation dates are shuffled with a fixed seed and dealt into five folds; fold 1 (141 images) is held out.
  The U-Net and the detectors train on the other 566 images. The rescorer and the refiner are fitted on the 141
  held-out images, because they learn from the first-stage models' errors on images those models have not seen.
- Retraining is not bit-reproducible (no global seed in two of the recipes, cuDNN / AMP, two GPU types); scores are
  expected to stay within the seed noise (about +-.005 PQ).
- The U-Net step runs the original training script unchanged and therefore needs Linux and an NVIDIA GPU.

In [4]:
# 2. training (optional): the recipe of every file in weights/, and retraining when TRAIN is True
for row in ft.describe():
    print(f"{row['step']:9s} -> {row['file']}")
    print(f"          data: {row['data']}")
    print(f"          settings: {row['settings']}")
    print(f"          original run: {row['original run']} ({row['runner']}; {row['record']})")
if TRAIN:
    t = time.perf_counter()
    gpu_index = int(DEVICE.split(':')[1]) if DEVICE.startswith('cuda:') else 0
    report['training'] = ft.retrain(TRAIN_STEPS, device=DEVICE, gpu_index=gpu_index, finals=RUN_FINALS, workers=PREPROCESS_WORKERS)
    report['timings']['training_s'] = time.perf_counter() - t
    fp.WEIGHTS = ft.OUT / 'weights'   # the stages below now read the retrained files
    print(f"retrained {list(report['training'])} in {report['timings']['training_s'] / 3600:.1f} h -> {fp.WEIGHTS}")
else:
    print('TRAIN = False: the stages below use weights/ (SHA-256 verified above)')

unet      -> unet_effb4_cpstrong_s0.pt
          data: 566 fold1 training images, one annotator per image per epoch, 512 px crops
          settings: EfficientNet-B4 U-Net (ImageNet encoder), 30 epochs, batch 16, AdamW 1e-3 cosine, AMP, boundary-weighted BCE (band 2 px, weight 4) + Dice, 1-3 pasted filaments per crop, last checkpoint
          original run: RTX A5000, time not recorded, peak 12.2 GB (scripts/train_unet_cpstrong_20261025.py --seed 0; docs/experiments/unet_copypaste_strong_20261025.md)
cp_s43    -> yolo11m_seg_cp_s43_1536.pt
          data: 566 training images, medoid annotator, raw JPEG
          settings: yolo11m-seg (COCO), imgsz 1536, 50 epochs, batch 2, mosaic .25, copy_paste .5, seed 43, best.pt
          original run: RTX A5000, 1 h, peak 12.1 GB (scripts/run_yolo_copypaste_20261025.py s43 train; docs/experiments/yolo_copypaste_detector_20261025.md)
m1280     -> yolo11m_seg_m1280.pt
          data: 566 training images, medoid annotator, raw JPEG
          settings

## 3. Inference

The two kinds of networks, each on every test image: the U-Net says which pixels are filament, the
detectors say which pixels belong to the same filament.

In [5]:
# 3a. U-Net logits (dihedral-8 TTA, 512 tiles, overlap 128) -> float16 per image
t = time.perf_counter()
report['stages']['unet'] = fp.unet_logits(stems, fp.WEIGHTS / fp.weight_entries()['unet']['file'], CACHE, LOGITS, torch.device(DEVICE))
report['timings']['unet_s'] = time.perf_counter() - t
print(report['stages']['unet'], f"{report['timings']['unet_s']:.0f}s = {report['timings']['unet_s'] / len(stems):.1f}s/image")

device: cuda:0


models: 1; computing logits for 180/180 observations


  5/180


  10/180


  15/180


  20/180


... (29 outputs left out)

  170/180


  175/180


  180/180


{'images': 180, 'tta': 'dihedral', 'tile': 512, 'overlap': 128, 'dtype': 'float16', 'checkpoint_sha256': '37fe7fb2753397fc0517d7f7036fb6efba15d81ea88efd134f7301bbdb2eab8a'} 446s = 2.5s/image


In [6]:
# 3b. detectors: four flipped views per image per member
t = time.perf_counter()
report['stages']['detectors'] = fp.detector_views(stems, members, IMAGE_DIR, CACHE, VIEWS, DEVICE)
report['timings']['detectors_s'] = time.perf_counter() - t
for m, r in report['stages']['detectors'].items():
    print(m, {k: v for k, v in r.items() if k in ('cached', 'seconds', 'detections_per_view')})
print(f"{report['timings']['detectors_s']:.0f}s = {report['timings']['detectors_s'] / len(stems):.1f}s/image for {len(members)} detectors")

  [cp_s43] 30/180 43s


  [cp_s43] 60/180 87s


  [cp_s43] 90/180 139s


  [cp_s43] 120/180 178s


  [cp_s43] 150/180 202s


  [cp_s43] 180/180 228s


... (26 outputs left out)

  [d1536] 90/180 122s


  [d1536] 120/180 155s


  [d1536] 150/180 175s


  [d1536] 180/180 198s
cp_s43 {'cached': False, 'seconds': 227.88179125497118, 'detections_per_view': [2617, 2588, 2602, 2555]}
m1280 {'cached': False, 'seconds': 195.1585105988197, 'detections_per_view': [2311, 2308, 2310, 2340]}
m1024 {'cached': False, 'seconds': 218.31272670719773, 'detections_per_view': [2471, 2485, 2463, 2485]}
det4 {'cached': False, 'seconds': 189.10980993136764, 'detections_per_view': [2223, 2247, 2226, 2213]}
d1024 {'cached': False, 'seconds': 173.5996619630605, 'detections_per_view': [2126, 2146, 2150, 2127]}
d1536 {'cached': False, 'seconds': 197.6348639028147, 'detections_per_view': [2214, 2208, 2264, 2257]}
1202s = 6.7s/image for 6 detectors


## 4. Post-processing

Per setting: detections claim the U-Net foreground pixels (fusion), the rescorer keeps the candidates whose
expected contribution to PQ is positive, the refiner redraws every kept instance, and the CSV is written and checked.

In [7]:
# 4. per setting: fusion + frozen rescorer -> refiner -> CSV -> comparison
manifest = fp.manifest()
for final in RUN_FINALS:
    pool = fp.FINALS[final]
    pool_members = fp.POOLS[pool]
    out = OUT / final
    out.mkdir(parents=True, exist_ok=True)
    rescorer = fp.FrozenRescorer.load(fp.WEIGHTS / manifest['finals'][final]['rescorer'])
    t = time.perf_counter()
    preds, rows = fp.select_instances(stems, pool_members, LOGITS, VIEWS, CACHE, rescorer)
    t_select = time.perf_counter() - t
    unrefined = fp.write_csv(preds, out / 'submission_unrefined.csv')
    t = time.perf_counter()
    refined, refine_info = fp.refine(preds, CACHE, fp.WEIGHTS / fp.weight_entries()['refiner']['file'], DEVICE, batch=REFINER_BATCH)
    t_refine = time.perf_counter() - t
    checks = fp.write_csv(refined, out / 'submission.csv')
    entry = {'pool': pool, 'detectors': list(pool_members), 'rescorer': rescorer.params['fit'],
             'candidates': sum(r['candidates'] for r in rows), 'kept': sum(r['kept'] for r in rows),
             'unrefined_csv': unrefined, 'refine': refine_info, 'csv': checks,
             'timings': {'select_s': t_select, 'refine_s': t_refine}}
    reference = ROOT / manifest['finals'][final]['csv']
    if COMPARE and reference.exists():
        entry['vs_uploaded_final'] = fp.compare_csv(out / 'submission.csv', reference, stems if LIMIT else None)
        entry['vs_uploaded_final']['expected_sha256'] = manifest['finals'][final]['csv_sha256']
    report['finals'][final] = entry
    print(final, {k: entry[k] for k in ('candidates', 'kept')}, 'refine', {k: refine_info[k] for k in ('instances', 'changed', 'fallback')},
          'csv', checks['csv_sha256'][:12], f'select {t_select:.0f}s refine {t_refine:.0f}s')
    if 'vs_uploaded_final' in entry:
        print('   vs uploaded:', {k: v for k, v in entry['vs_uploaded_final'].items() if k not in ('reference', 'reference_sha256')})

  select 30/180 candidates=279 kept=217


  select 60/180 candidates=553 kept=432


  select 90/180 candidates=899 kept=718


  select 120/180 candidates=1141 kept=910


  select 150/180 candidates=1260 kept=1007


  select 180/180 candidates=1398 kept=1108


... (6 outputs left out)

  refine 140/178 348s


  refine 160/178 367s


  refine 178/178 395s


final1_A6 {'candidates': 1398, 'kept': 1108} refine {'instances': 1108, 'changed': 1108, 'fallback': 0} csv 01ee0993a800 select 68s refine 396s
   vs uploaded: {'csv_sha256': '01ee0993a80056119a5d16410124e30c72285cbc699133aeb67cd171a005a256', 'byte_equal': False, 'reference_instances': 1109, 'instances': 1108, 'exact_rle_matches': 1105, 'matched_iou_gt_05': 1108, 'match_share': 0.9990982867448152, 'images_with_different_count': 1, 'expected_sha256': '2ba22b868843cfb90c9216451961ae33d58f22208014fe077b5f22b4ab0d7a46'}


## 5. Evaluation

Test images have no annotations, so the pipeline is scored on the 141 held-out validation images (228 annotator
records). The four quantities are the ones in the competition rubric, with the definitions of the official
self-evaluation notebook: every annotator's record is scored separately; a hit needs IoU > .5; the IoU and Dice
distributions use every overlapping annotation-prediction pair; a partner is any overlapping mask of the other side.

- **Recorded cross-fitted predictions (default, CPU, seconds).** `docs/report/val_predictions/` holds the pipeline's
  predictions on these images. The U-Net and the detectors never saw them; the rescorer and the refiner were fitted on
  one date half and applied to the other, so no stage is scored on images it learned from. The cell re-scores them and
  stops if the result differs from the recorded score.
- **Rerun (`EVALUATE_RERUN = True`, GPU).** Runs the weights loaded above on the same images and scores the result.
  The shipped rescorer and refiner were fitted on all 141 images, so this number is in-sample for those two stages and
  is slightly optimistic; it checks that the code path works, the cross-fitted number is the one to report.

In [8]:
# 5. evaluation on the 141 held-out validation images: PQ, IoU / Dice distributions, one-to-many / many-to-one
store = fe.AnnotationStore(fe.DEFAULT_DATA)


def show(name, r):
    pairs, human = r['prediction_vs_annotation'], r['annotator_vs_annotator']
    print(f"{name}: PQ {r['pq']:.4f} (SQ {r['sq']:.4f}, RQ {r['rq']:.4f}; TP {r['tp']}, FP {r['fp']}, FN {r['fn']}) "
          f"on {r['images']} images / {r['annotator_records']} annotator records")
    if pairs['pairs']:
        print(f"   overlapping pairs {pairs['pairs']}: IoU mean {pairs['iou_mean']:.3f}, median {pairs['iou_median']:.3f}; "
              f"Dice mean {pairs['dice_mean']:.3f}, median {pairs['dice_median']:.3f}")
    if human['pairs']:
        print(f"   annotator vs annotator ({human['images']} images, {human['pairs']} pairs): IoU mean {human['iou_mean']:.3f}, "
              f"Dice mean {human['dice_mean']:.3f}")
    print(f"   annotations by number of overlapping predictions {r['annotation_partners']} "
          f"(one-to-many: {r['one_to_many_annotations']}); predictions by number of overlapping annotations "
          f"{r['prediction_partners']} (many-to-one: {r['many_to_one_predictions']})")


report['evaluation'] = {}
for final in RUN_FINALS:
    setting = fp.FINALS[final]
    for stage, key in (('before the refiner', 'original_rle'), ('after the refiner', 'rle')):
        result = fe.summary(fe.load_predictions(fe.VAL_PREDICTIONS[setting], key), store)
        fe.check_recorded(result, setting, key)
        report['evaluation'][f'{final} cross-fitted, {stage}'] = result
        show(f'{final} cross-fitted, {stage}', result)

final1_A6 cross-fitted, before the refiner: PQ 0.4639 (SQ 0.6858, RQ 0.6765; TP 1053, FP 415, FN 592) on 141 images / 228 annotator records
   overlapping pairs 1375: IoU mean 0.599, median 0.651; Dice mean 0.728, median 0.789
   annotator vs annotator (61 images, 663 pairs): IoU mean 0.523, Dice mean 0.664
   annotations by number of overlapping predictions {0: 333, 1: 1256, 2: 49, 3: 7} (one-to-many: 56); predictions by number of overlapping annotations {0: 162, 1: 1242, 2: 59, 3: 5} (many-to-one: 64)
final1_A6 cross-fitted, after the refiner: PQ 0.4670 (SQ 0.6877, RQ 0.6791; TP 1057, FP 411, FN 588) on 141 images / 228 annotator records
   overlapping pairs 1368: IoU mean 0.604, median 0.657; Dice mean 0.733, median 0.793
   annotator vs annotator (61 images, 663 pairs): IoU mean 0.523, Dice mean 0.664
   annotations by number of overlapping predictions {0: 333, 1: 1263, 2: 42, 3: 7} (one-to-many: 49); predictions by number of overlapping annotations {0: 164, 1: 1245, 2: 54, 3: 5} (

In [9]:
# 5 (optional). rerun on the validation images with the weights loaded above (in-sample for rescorer and refiner)
if EVALUATE_RERUN:
    _, val_stems = ft.fold1_split(ft.annotation_stems(fp.DEFAULT_DATA))
    if LIMIT:
        val_stems = val_stems[:LIMIT]
    for final in RUN_FINALS:
        t = time.perf_counter()
        refined, unrefined = fp.predict(val_stems, fp.DEFAULT_DATA / 'train/train_images', OUT / 'validation', final, DEVICE,
                                        workers=PREPROCESS_WORKERS, refiner_batch=REFINER_BATCH)
        report['timings'][f'validation_rerun_{final}_s'] = time.perf_counter() - t
        for stage, preds in (('before the refiner', unrefined), ('after the refiner', refined)):
            result = fe.summary(preds, store)
            report['evaluation'][f'{final} rerun, {stage}'] = result
            show(f'{final} rerun, {stage}', result)
else:
    print('EVALUATE_RERUN = False: scored the recorded cross-fitted predictions only')

device: cuda:0


models: 1; computing logits for 141/141 observations


  5/141


  10/141


  15/141


  20/141


... (64 outputs left out)

  refine 100/137 281s


  refine 120/137 315s


  refine 137/137 332s


final1_A6 rerun, before the refiner: PQ 0.4673 (SQ 0.6854, RQ 0.6819; TP 1061, FP 406, FN 584) on 141 images / 228 annotator records
   overlapping pairs 1379: IoU mean 0.600, median 0.652; Dice mean 0.729, median 0.789
   annotator vs annotator (61 images, 663 pairs): IoU mean 0.523, Dice mean 0.664
   annotations by number of overlapping predictions {0: 328, 1: 1261, 2: 50, 3: 6} (one-to-many: 56); predictions by number of overlapping annotations {0: 155, 1: 1250, 2: 57, 3: 5} (many-to-one: 62)
final1_A6 rerun, after the refiner: PQ 0.4741 (SQ 0.6901, RQ 0.6870; TP 1069, FP 398, FN 576) on 141 images / 228 annotator records
   overlapping pairs 1364: IoU mean 0.611, median 0.662; Dice mean 0.740, median 0.796
   annotator vs annotator (61 images, 663 pairs): IoU mean 0.523, Dice mean 0.664
   annotations by number of overlapping predictions {0: 331, 1: 1270, 2: 38, 3: 6} (one-to-many: 44); predictions by number of overlapping annotations {0: 156, 1: 1263, 2: 43, 3: 5} (many-to-one: 4

## Summary

Time per stage and for the whole run; everything above is also written to `report.json`.

In [10]:
# Summary: time per image and the whole run; report.json next to the CSVs
n = len(stems)
tot = sum(report['timings'].values()) + sum(sum(e['timings'].values()) for e in report['finals'].values())
report['timings']['total_s'] = tot
report['finished'] = time.strftime('%Y-%m-%d %H:%M:%S')
print(f"device {DEVICE} ({report['environment'].get('gpu', 'cpu')}); {n} images")
print(f"preprocess {report['timings']['preprocess_s']:.0f}s | unet {report['timings']['unet_s']:.0f}s ({report['timings']['unet_s'] / n:.1f}s/img) | "
      f"detectors {report['timings']['detectors_s']:.0f}s ({report['timings']['detectors_s'] / n:.1f}s/img)")
for final, e in report['finals'].items():
    print(f"{final}: select {e['timings']['select_s']:.0f}s, refine {e['timings']['refine_s']:.0f}s "
          f"({e['timings']['refine_s'] / max(e['kept'], 1):.2f}s/instance), csv {e['csv']['csv_sha256'][:12]}")
print(f'total {tot:.0f}s = {tot / 60:.1f} min')
(OUT / 'report.json').write_text(json.dumps(report, indent=1, default=str), encoding='utf-8')
print('wrote', OUT / 'report.json')

device cuda:0 (NVIDIA RTX A5000); 180 images
preprocess 29s | unet 446s (2.5s/img) | detectors 1202s (6.7s/img)
final1_A6: select 68s, refine 396s (0.36s/instance), csv 01ee0993a800
total 3858s = 64.3 min
wrote /home/<user>/solar_filament_2026/final_check_20261112/repo/outputs/final_inference/report.json
